# Task

With AI-modelling there is plenty of opportunity to improve processes or suggest improved ways of doing things. When doing so it is often very smart and efficient (time is a scarce resource) to create a POC (Proof of Concept) which basically is a small demo checking wether it is worthwile going further with something. It is also something concrete which facilitates discussions, do not underestimate the power of that. 

In this example, you are working in a company that sells houses and they have a "manual" process of setting prices by humans. You as a AI-modeller can make this process better by using Machine Learning. Your task is to create a POC that you will present to your team colleagues and use as a source of discussion of wether or not you should continue with more detailed modelling. 

Two quotes to facilitate your reflection on the value of creating a PoC: 

"*Premature optimization is the root of all evil*". 

"*Fail fast*".


**More specifially, do the following:**
1. A short EDA (Exploratory Data Analysis) of the housing data set.
2. Drop the column "ocean_proximity", then you only have numeric columns which will simplify your analysis. Remember, this is a POC!
3. Split your data into train, validation and test set. Before this, split your data into y and X. 
4. You have missing values in your data (not sure you do but you can assume so). Handle this with [ SimpleImputer(strategy="median") ], check the fantastic Scikit-learn documentation for details. Notice, the SimpleImputer should only be used for transformation on the validation and test data. Not fitting. 
5. Create one "Linear Regression" model and one "Lasso" model. For the Lasso model, use GridSearchCV to optimize $\\alpha$ values, choose yourself which $\\alpha$ values to evaluate.
Use RMSE as a metric to decide which model to choose. 

7. Which model is best on the validation data? 

8. Evaluate your chosen model on the test set using the root mean squared error (RMSE) as the metric. Conclusions? To be 100% right, you should re-fit your chosed model on the combination of train+val data. 

8. Do a short presentation (~ 2-5 min) on your POC that you present to your colleagues (no need to prepare anything particular, just talk from the code). Think of:
- What do you want to highlight/present?
- What is your conclusion?
- What could be the next step? Is the POC convincing enough or is it not worthwile continuing? Do we need to dig deeper into this before taking some decisions?

# Code

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import root_mean_squared_error

In [6]:
# Load data - works on both Google Colab and local Jupyter
housing = pd.read_csv('housing.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/content/housing.csv'

## 1. EDA

In [ ]:
housing.head()

In [ ]:
housing.info()

In [ ]:
housing.describe()

In [ ]:
print("Missing values per column:")
print(housing.isnull().sum())

In [ ]:
housing.hist(bins=50, figsize=(15, 10))
plt.tight_layout()
plt.show()

## 2. Drop ocean_proximity & prepare data

In [ ]:
# Drop non-numeric column
data = housing.drop(columns=['ocean_proximity'])

## 3. Split into X, y and train/val/test sets

In [ ]:
# Split data into features and target
X = data.drop(columns=['median_house_value'])
y = data['median_house_value']

In [ ]:
# First split: 80% train+val, 20% test
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=40)
# Second split: 70% train, 30% val (of the 80%)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.3, random_state=36)

print(f"Train size:      {X_train.shape}")
print(f"Validation size: {X_val.shape}")
print(f"Test size:       {X_test.shape}")

## 4. Handle missing values with SimpleImputer

Fit the imputer **only** on training data, then transform train, val, and test.

In [ ]:
imputer = SimpleImputer(strategy='median')

# Fit on train, transform all splits
X_train_imp = imputer.fit_transform(X_train)
X_val_imp   = imputer.transform(X_val)       # transform only, no fit
X_test_imp  = imputer.transform(X_test)      # transform only, no fit

print("Missing values after imputation (train):", np.isnan(X_train_imp).sum())

## 5. Train models: Linear Regression and Lasso (with GridSearchCV)

In [ ]:
# --- Linear Regression ---
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

lr_pipeline.fit(X_train_imp, y_train)

y_val_pred_lr = lr_pipeline.predict(X_val_imp)
rmse_lr_val = root_mean_squared_error(y_val, y_val_pred_lr)
print(f"Linear Regression - Validation RMSE: {rmse_lr_val:,.0f}")

In [ ]:
# --- Lasso with GridSearchCV ---
lasso_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Lasso(max_iter=10000))
])

param_grid = {
    'model__alpha': [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
}

grid_search = GridSearchCV(
    lasso_pipeline,
    param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

grid_search.fit(X_train_imp, y_train)

best_alpha = grid_search.best_params_['model__alpha']
print(f"Best alpha: {best_alpha}")

y_val_pred_lasso = grid_search.predict(X_val_imp)
rmse_lasso_val = root_mean_squared_error(y_val, y_val_pred_lasso)
print(f"Lasso (best alpha={best_alpha}) - Validation RMSE: {rmse_lasso_val:,.0f}")

## 6 & 7. Compare models on validation data

In [ ]:
print("=== Validation RMSE Comparison ===")
print(f"Linear Regression: {rmse_lr_val:,.0f}")
print(f"Lasso (alpha={best_alpha}): {rmse_lasso_val:,.0f}")

if rmse_lr_val < rmse_lasso_val:
    print("\n=> Best model on validation: Linear Regression")
    best_model_name = 'Linear Regression'
else:
    print(f"\n=> Best model on validation: Lasso (alpha={best_alpha})")
    best_model_name = 'Lasso'

## 8. Evaluate best model on test set

Re-fit the chosen model on train + validation data before evaluating on test.

In [ ]:
# Combine train + val for final training
X_trainval = np.vstack([X_train_imp, X_val_imp])
y_trainval = pd.concat([y_train, y_val])

if best_model_name == 'Linear Regression':
    final_model = Pipeline([
        ('scaler', StandardScaler()),
        ('model', LinearRegression())
    ])
else:
    final_model = Pipeline([
        ('scaler', StandardScaler()),
        ('model', Lasso(alpha=best_alpha, max_iter=10000))
    ])

final_model.fit(X_trainval, y_trainval)

y_test_pred = final_model.predict(X_test_imp)
rmse_test = root_mean_squared_error(y_test, y_test_pred)

print(f"=== Test Set Evaluation ===")
print(f"Best model: {best_model_name}")
print(f"Test RMSE: {rmse_test:,.0f}")
print(f"\nConclusion: The model predicts median house value with an average error of ~${rmse_test:,.0f}.")
print("This POC shows that a simple linear model can capture the main patterns in the data.")
print("Next steps could include: feature engineering, trying non-linear models (e.g. Random Forest),")
print("and including the ocean_proximity feature via one-hot encoding.")